<a href="https://colab.research.google.com/github/shivansh2310/Quantitative-Portfolio-Management/blob/main/Constrained_Portfolio_Optimization_(Chapter_8).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### A. The Objective Function with Friction

Traditional Markowitz optimization maximizes Alpha and penalizes Risk:
* $Max: w^T \mu - \lambda (w^T \Sigma w)$

Isichenko argues this is incomplete. You must include trading friction directly inside the optimizer's brain, or it will trade itself to death. The true StatArb objective function is:

* $Max: \text{Alpha} - \text{Risk Penalty} - \text{Cost Penalty}$
* $Max: w^T \mu - \frac{\lambda}{2} (w^T \Sigma w) - \text{Impact}(w)$

### B. The Constraints

We are building a Market Neutral Statistical Arbitrage portfolio.
1. Dollar Neutrality: The sum of the weights must equal 0 ($\sum w_i = 0$). We fund our longs entirely with our shorts.

2. Gross Leverage Bound: The absolute sum of the weights must equal 1 ($\sum |w_i| = 1.0$). We don't want the optimizer to demand $10x leverage to chase a tiny edge.

## Mean-Variance-Cost Optimizer

In [36]:
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.linalg import eigh
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

In [19]:
universe = {
    'AAPL': 'Tech', 'MSFT': 'Tech', 'NVDA': 'Tech', 'AMD': 'Tech', 'ORCL': 'Tech',
    'JPM': 'Fin', 'BAC': 'Fin', 'GS': 'Fin', 'MS': 'Fin', 'C': 'Fin',
    'XOM': 'Energy', 'CVX': 'Energy', 'COP': 'Energy', 'EOG': 'Energy', 'SLB': 'Energy',
    'JNJ': 'Health', 'UNH': 'Health', 'PFE': 'Health', 'ABBV': 'Health', 'MRK': 'Health'
}
tickers = list(universe.keys())

print("Fetching historical data... (Takes ~10 seconds)")
# Fetch 1 year of daily close prices
prices = yf.download(tickers, period="1y")['Close']
prices = prices[tickers] # Ensure column order

[                       0%                       ]

Fetching historical data... (Takes ~10 seconds)


[*********************100%***********************]  20 of 20 completed


In [20]:
# We use .shift(-1) because today's features must predict tomorrow's return
raw_returns = prices.pct_change()
forward_returns = raw_returns.shift(-1)

In [21]:
# Convert to a "Long" format DataFrame (Standard for ML pipelines)
df = forward_returns.unstack().reset_index()
df.columns = ['Ticker', 'Date', 'Fwd_Return']
df = df.dropna()


In [22]:
# Map the sectors
df['Sector'] = df['Ticker'].map(universe)

In [23]:
def neutralize_and_rank(daily_data):
    # Market Neutralization (Subtract cross-sectional mean)
    daily_data['Market_Mean'] = daily_data['Fwd_Return'].mean()
    daily_data['Market_Neutral'] = daily_data['Fwd_Return'] - daily_data['Market_Mean']

    # Sector Neutralization (Subtract sector mean from the market-neutral returns)
    sector_means = daily_data.groupby('Sector')['Market_Neutral'].transform('mean')
    daily_data['Idiosyncratic_Return'] = daily_data['Market_Neutral'] - sector_means

    # Rank Normalization (Scale between 0 and 1 to suppress outliers)
    daily_data['ML_Target'] = daily_data['Idiosyncratic_Return'].rank(pct=True)

    return daily_data

print("Applying Cross-Sectional Neutralization and Rank Normalization...")
# Apply the function group-by-group for every single day in the dataset
ml_dataset = df.groupby('Date', group_keys=False).apply(neutralize_and_rank)


Applying Cross-Sectional Neutralization and Rank Normalization...


In [24]:
# We must sort by Ticker and Date to calculate rolling features correctly
ml_dataset = ml_dataset.sort_values(['Ticker', 'Date'])

In [25]:

# Short-Term Mean Reversion (5-Day Return)
# Hypothesis: High 5-day return means it's overbought and will revert (negative weight expected)
ml_dataset['F_Reversion_5d'] = ml_dataset.groupby('Ticker')['Fwd_Return'].transform(lambda x: x.shift(1).rolling(5).sum())

# Medium-Term Momentum (21-Day Return)
# Hypothesis: High 1-month return means it's trending (positive weight expected)
ml_dataset['F_Momentum_1m'] = ml_dataset.groupby('Ticker')['Fwd_Return'].transform(lambda x: x.shift(1).rolling(21).sum())

# Daily Volatility (21-Day standard deviation)
ml_dataset['F_Volatility'] = ml_dataset.groupby('Ticker')['Fwd_Return'].transform(lambda x: x.shift(1).rolling(21).std())

ml_dataset = ml_dataset.dropna()

print("Cross-Sectional Feature Standardization (Z-Scoring)...")
# Just like targets, features must be standardized cross-sectionally every single day
features = ['F_Reversion_5d', 'F_Momentum_1m', 'F_Volatility']


Cross-Sectional Feature Standardization (Z-Scoring)...


In [26]:
def standardize_features(daily_data):
    for f in features:
        daily_data[f] = (daily_data[f] - daily_data[f].mean()) / (daily_data[f].std() + 1e-8)
    return daily_data

ml_dataset = ml_dataset.groupby('Date', group_keys=False).apply(standardize_features)

In [27]:
print("Initializing the XGBoost Regressor...")
# MFE Constraints for Financial Data:
# 1. max_depth=3 (Prevent memorizing noise)
# 2. learning_rate=0.05 (Learn slowly)
# 3. subsample=0.8 (Train on 80% of data per tree to prevent overfitting)
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror'
)

Initializing the XGBoost Regressor...


In [28]:
# X is our features, y is our Neutralized Rank Target from Day 1
X = ml_dataset[features]
y = ml_dataset['ML_Target']

In [29]:
print("Training the Non-Linear Ensemble...")
xgb_model.fit(X, y)

Training the Non-Linear Ensemble...


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [30]:
tickers = list(ml_dataset['Ticker'].unique())
raw_data = yf.download(tickers, period="3mo")

# 1. Calculate Raw 21-day Average Daily Dollar Volume (ADV)
close_prices = raw_data['Close']
volumes = raw_data['Volume']
dollar_volumes = close_prices * volumes
adv_21d = dollar_volumes.rolling(21).mean().iloc[-1] # Latest ADV

# 2. Calculate RAW 21-day Volatility (The Fix)
# We calculate the actual standard deviation of daily returns
recent_returns = close_prices.pct_change()
raw_volatility_21d = recent_returns.rolling(21).std().iloc[-1]
volatility = raw_volatility_21d # Assign the raw volatility to the expected variable name

# Get the theoretical XGBoost Alpha (Prediction)
ml_dataset['XGB_Prediction'] = xgb_model.predict(X)
latest_date = ml_dataset['Date'].max()
latest_data = ml_dataset[ml_dataset['Date'] == latest_date].set_index('Ticker')
xgb_alpha = latest_data['XGB_Prediction']

[*********************100%***********************]  20 of 20 completed


In [33]:
print("Preparing the Return Matrix...")
returns_matrix = raw_returns.dropna().values
T, N = returns_matrix.shape

Preparing the Return Matrix...


In [34]:
# Standardize returns (mean 0, variance 1) for PCA
returns_std = (returns_matrix - np.mean(returns_matrix, axis=0)) / np.std(returns_matrix, axis=0)

In [37]:
# Calculate the empirical correlation matrix
corr_matrix = np.corrcoef(returns_std, rowvar=False)

# Extract eigenvalues and eigenvectors
eigenvalues, eigenvectors = eigh(corr_matrix)

# Sort them in descending order
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

In [38]:
print("Applying Random Matrix Theory (Marchenko-Pastur)...")
# Calculate the Marchenko-Pastur maximum theoretical noise bound
q = T / N  # Ratio of time observations to assets
sigma_sq = 1.0 # Variance is 1 because we standardized
mp_max = sigma_sq * (1 + np.sqrt(1/q))**2

print(f"Marchenko-Pastur Theoretical Maximum Noise Bound: {mp_max:.4f}")

Applying Random Matrix Theory (Marchenko-Pastur)...
Marchenko-Pastur Theoretical Maximum Noise Bound: 1.6442


In [39]:
# Find how many factors are real signal (above the noise bound)
num_signals = len(eigenvalues[eigenvalues > mp_max])
print(f"Number of True Structural Factors discovered: {num_signals} out of {N}")

Number of True Structural Factors discovered: 3 out of 20


In [40]:
print("\nClipping the Noise...")
# Keep the signal eigenvalues
clipped_eigenvalues = eigenvalues.copy()

# Average the noise eigenvalues to preserve the trace (sum of variance)
noise_average = np.mean(eigenvalues[num_signals:])
clipped_eigenvalues[num_signals:] = noise_average

# Reconstruct the cleaned correlation matrix
cleaned_corr_matrix = eigenvectors @ np.diag(clipped_eigenvalues) @ eigenvectors.T

# Convert back to a covariance matrix using the original standard deviations
stds = np.std(returns_matrix, axis=0)
cleaned_cov_matrix = np.outer(stds, stds) * cleaned_corr_matrix

print("STATUS: RMT Covariance Matrix successfully reconstructed and cleaned of noise.")


Clipping the Noise...
STATUS: RMT Covariance Matrix successfully reconstructed and cleaned of noise.


In [43]:
print("Assembling the Optimizer Inputs...")

# 1. The Alpha Vector (mu)
raw_ranks = latest_data['XGB_Prediction'].values
alpha_vector = raw_ranks - np.mean(raw_ranks)

# 2. The Risk Matrix (Sigma)
cov_matrix = cleaned_cov_matrix * 252

# 3. Align ADV and Volatility vectors for the Cost Penalty
# We fill missing values with massive liquidity to prevent math errors
adv_vector = np.array([adv_21d.get(t, 1e10) for t in tickers])
vol_vector = np.array([raw_volatility_21d.get(t, 0.02) for t in tickers])

risk_aversion = 2.5
PORTFOLIO_AUM = 100_000_000 # $100M

Assembling the Optimizer Inputs...


In [42]:
print("Defining the StatArb Objective Function...")

def objective_function(weights, alpha, cov, risk_aversion):
    # Expected Portfolio Return
    port_alpha = np.dot(weights, alpha)

    # Expected Portfolio Variance
    port_variance = np.dot(weights.T, np.dot(cov, weights))

    # We want to MAXIMIZE alpha and MINIMIZE variance.
    # Scipy only MINIMIZES, so we return the negative of the utility function.
    utility = port_alpha - (risk_aversion / 2) * port_variance
    return -utility

Defining the StatArb Objective Function...


In [44]:
print("Defining the Mean-Variance-COST Objective Function...")

# Notice the new arguments: adv, vol, and aum
def objective_function(weights, alpha, cov, risk_aversion, adv, vol, aum):
    # Expected Portfolio Return
    port_alpha = np.dot(weights, alpha)

    # Expected Portfolio Variance
    port_variance = np.dot(weights.T, np.dot(cov, weights))

    # --- THE FIX: Dynamic Market Impact Penalty ---
    c_constant = 0.1
    # Calculate the dollar size of the order for each asset
    order_sizes = np.abs(weights) * aum

    # Calculate the slippage (in decimal form) for each asset dynamically
    impact_costs = c_constant * vol * np.sqrt(order_sizes / adv)

    # Total drag on portfolio return = sum of (weight * slippage)
    total_impact_penalty = np.sum(np.abs(weights) * impact_costs)
    # ----------------------------------------------

    # The True StatArb Utility: Maximize Alpha, Minimize Risk, Minimize Friction
    utility = port_alpha - ((risk_aversion / 2) * port_variance) - total_impact_penalty

    # Return negative utility because scipy only minimizes
    return -utility

Defining the Mean-Variance-COST Objective Function...


In [45]:
print("Defining Constraints (Market Neutrality)...")

def dollar_neutral_constraint(weights):
    return np.sum(weights)

def leverage_constraint(weights):
    return np.sum(np.abs(weights)) - 1.0

constraints = [
    {'type': 'eq', 'fun': dollar_neutral_constraint},
    {'type': 'eq', 'fun': leverage_constraint}
]

bounds = tuple((-0.10, 0.10) for _ in range(N))
init_weights = np.sign(alpha_vector)
init_weights = init_weights / np.sum(np.abs(init_weights))

Defining Constraints (Market Neutrality)...


In [46]:
print("Running the Convex Optimizer...")
result = minimize(
    objective_function,
    init_weights,
    args=(alpha_vector, cov_matrix, risk_aversion, adv_vector, vol_vector, PORTFOLIO_AUM),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    options={'maxiter': 1000, 'ftol': 1e-9}
)

if result.success:
    print("STATUS: Optimization Converged Successfully.\n")
    optimal_weights = pd.Series(result.x, index=tickers).sort_values(ascending=False)

    print("="*50)
    print(" OPTIMAL TARGET PORTFOLIO WEIGHTS")
    print("="*50)
    print("TOP 3 LONGS:")
    print(optimal_weights.head(3).round(4) * 100)
    print("\nTOP 3 SHORTS:")
    print(optimal_weights.tail(3).round(4) * 100)

    print("-" * 50)
    print(f"Net Exposure:  {optimal_weights.sum():.6f}")
    print(f"Gross Leverage: {optimal_weights.abs().sum():.4f}")
    print("="*50)
else:
    print("STATUS: Optimization Failed.")
    print(result.message)

Running the Convex Optimizer...
STATUS: Optimization Converged Successfully.

 OPTIMAL TARGET PORTFOLIO WEIGHTS
TOP 3 LONGS:
AAPL    10.0
AMD     10.0
NVDA    10.0
dtype: float64

TOP 3 SHORTS:
JNJ   -10.0
MRK   -10.0
XOM   -10.0
dtype: float64
--------------------------------------------------
Net Exposure:  0.000000
Gross Leverage: 1.0000
